In [1]:
import os, sys
import torch

PROJECT = os.path.expanduser("~/spline_flows")
sys.path.insert(0, PROJECT)
from conditional_unet1d import ConditionalUnet1D

HIST, J = 8, 29   # 8 history frames, 29 joints

def make_model():
    return ConditionalUnet1D(
        input_dim=J,                    # channels = joints
        global_cond_dim=HIST * J,       # flattened history: 232 numbers
        diffusion_step_embed_dim=128,   # size of the t embedding
        down_dims=[128, 256, 512],      # channels at each of the 3 levels
        kernel_size=5,                  # each conv looks at 5 neighbouring time steps
        n_groups=8,                     # GroupNorm groups
        cond_predict_scale=True,        # FiLM: condition gives scale AND shift
    )

B = 4                                   # fake batch of 4
history = torch.randn(B, HIST, J)       # random stand-in for real history
t = torch.randint(0, 100, (B,))         # a random noise level per sample

for name, length in [("spline (K=14)", 16), ("waypoint", 32)]:
    model = make_model()
    n_params = sum(p.numel() for p in model.parameters())
    x = torch.randn(B, length, J)       # stand-in for a noisy target
    out = model(x, t, global_cond=history.flatten(start_dim=1))
    print(f"{name:14s} in {tuple(x.shape)} -> out {tuple(out.shape)}   params {n_params:,}")

try:
    make_model()(torch.randn(B, 6, J), t, global_cond=history.flatten(start_dim=1))
except RuntimeError as e:
    print("length 6 (K=4) fails:", str(e).splitlines()[0])

spline (K=14)  in (4, 16, 29) -> out (4, 16, 29)   params 18,006,429
waypoint       in (4, 32, 29) -> out (4, 32, 29)   params 18,006,429
length 6 (K=4) fails: Sizes of tensors must match except in dimension 1. Expected size 4 but got size 3 for tensor number 1 in the list.


In [2]:
import numpy as np
from quadratic_spline import QuadraticSpline

K, H = 14, 32          # segments, frames in the future window
P = K + 2              # spline parameters per joint (16)
FPS = 30

d = np.load(os.path.join(PROJECT, "walk_windows_history8_horizon32_stride4.npz"))
J_min, J_max = d["J_min"], d["J_max"]
def normalize(x):   return 2 * (x - J_min) / (J_max - J_min + 1e-6) - 1
def unnormalize(z): return (z + 1) / 2 * (J_max - J_min + 1e-6) + J_min

spline = QuadraticSpline(nbSeg=K, nbDim=J, device="cpu")

# Basis matrix: how much each of the 16 parameters contributes to each of the 32 frames
t = torch.linspace(0, 1, H)
Psi, _, _ = spline.computePsiList1D(t)        # (32*29, 16*29) = (928, 464)

def encode(Y):
    """(B, 32, 29) radians -> spline params (B, 16, 29) in normalized units, plus the fitted curve in radians"""
    Yn = torch.tensor(normalize(Y), dtype=torch.float32)
    w, recon_n = spline.encode_trajectory(Yn)  # w comes out flat: (B, 464)
    return w.reshape(-1, P, J), unnormalize(recon_n.numpy())

def decode(W):
    """(B, 16, 29) normalized params -> (B, 32, 29) normalized trajectory"""
    B = W.shape[0]
    return (W.reshape(B, -1) @ Psi.T).reshape(B, H, J)

In [3]:
W_train, recon_train = encode(d["Y_train"])
W_test,  recon_test  = encode(d["Y_test"])
print(W_train.shape, W_test.shape)

# 1. Decode must reproduce the repo's fitted curve exactly
roundtrip = unnormalize(decode(W_train).numpy())
print("max round-trip difference (rad):", np.abs(roundtrip - recon_train).max())

# 2. Fit floor, now on held-out windows too
def errors(pred, true):
    pos = np.sqrt(np.mean((pred - true) ** 2))
    vel = np.sqrt(np.mean((np.diff(pred, axis=1) * FPS - np.diff(true, axis=1) * FPS) ** 2))
    return round(float(pos), 4), round(float(vel), 3)
print("train fit floor (pos rad, vel rad/s):", errors(recon_train, d["Y_train"]))
print("test  fit floor (pos rad, vel rad/s):", errors(recon_test,  d["Y_test"]))

# 3. Range of what each model will learn to generate
Wn = W_train.numpy()
print(f"spline targets:   min {Wn.min():.3f}  max {Wn.max():.3f}  outside [-1, 1]: {100*np.mean(np.abs(Wn) > 1):.3f}%")
print("largest |value| at each of the 16 positions:", np.round(np.abs(Wn).max(axis=(0, 2)), 2))
Yn_train = normalize(d["Y_train"])
print(f"waypoint targets: min {Yn_train.min():.3f}  max {Yn_train.max():.3f}")

np.savez(os.path.join(PROJECT, "spline_targets_K14.npz"),
         W_train=Wn, W_test=W_test.numpy(), K=K)

torch.Size([17813, 16, 29]) torch.Size([3791, 16, 29])
max round-trip difference (rad): 0.0
train fit floor (pos rad, vel rad/s): (0.0043, 0.204)
test  fit floor (pos rad, vel rad/s): (0.0049, 0.232)
spline targets:   min -1.715  max 1.670  outside [-1, 1]: 0.085%
largest |value| at each of the 16 positions: [1.06 1.71 1.1  1.12 1.12 1.13 1.14 1.13 1.12 1.15 1.15 1.17 1.13 1.13
 1.1  1.07]
waypoint targets: min -1.000  max 1.000


In [4]:
import sys
!{sys.executable} train.py --target spline   --max_steps 30 --batch_size 64 --out runs_smoke
!{sys.executable} train.py --target waypoint --max_steps 30 --batch_size 64 --out runs_smoke

/home/eychung/.conda/envs/spline/lib/python3.11/site-packages/torch/cuda/__init__.py:241: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
target=spline  train (17813, 16, 29)  test (3791, 16, 29)  device=cpu
params 18,006,429  steps/epoch 279  total steps 30
epoch    1  step     30  train 1.0342  val(ema) 1.0063  0.2 min
quick check on 256 test windows: position 3.6400 rad, velocity 131.053 rad/s, sampling 11.60 s for the batch
saved to runs_smoke/spline
/home/eychung/.conda/envs/spline/lib/python3.11/site-packages/torch/cuda/__init__.py:241: UserWarning:

In [5]:
import common as C

d = np.load(C.DATA)
is_val, names, file_id = C.split_train_val(d)
for k, name in enumerate(names):
    n_frames = sum(1 for _ in open(os.path.expanduser(f"~/data/LAFAN1_g1/g1/{name}.csv")))
    n_win = np.sum(file_id == k)
    tag = "VAL" if is_val[file_id == k].all() else ""
    print(f"{name:16s} frames {n_frames:6d}  expected windows {(n_frames - 40) // 4 + 1:5d}  found {n_win:5d}  {tag}")
print("validation windows:", is_val.sum(), "of", len(is_val))

/home/eychung/.conda/envs/spline/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/eychung/.conda/envs/spline/lib/python3.11/site-packages/torch/cuda/__init__.py:241: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


walk1_subject1   frames   7840  expected windows  1951  found  1951  
walk1_subject5   frames   7840  expected windows  1951  found  1951  
walk2_subject1   frames   7146  expected windows  1777  found  1777  
walk2_subject3   frames   7146  expected windows  1777  found  1777  
walk2_subject4   frames   7146  expected windows  1777  found  1777  VAL
walk3_subject1   frames   7399  expected windows  1840  found  1840  
walk3_subject3   frames   7399  expected windows  1840  found  1840  
walk3_subject4   frames   7399  expected windows  1840  found  1840  VAL
walk3_subject5   frames   7399  expected windows  1840  found  1840  
walk4_subject1   frames   4918  expected windows  1220  found  1220  
validation windows: 3617 of 17813


In [6]:
def load_joints(name):
    return np.loadtxt(os.path.expanduser(f"~/data/LAFAN1_g1/g1/{name}.csv"), delimiter=",")[:, 7:]

pairs = [("walk3_subject1", "walk3_subject2"),   # train vs test, same take
         ("walk1_subject1", "walk1_subject2"),   # train vs test, same take
         ("walk3_subject1", "walk3_subject4"),   # train vs validation, same take
         ("walk2_subject1", "walk2_subject3")]   # train vs train, same take
for a, b in pairs:
    A, B = load_joints(a), load_joints(b)
    diff = np.abs(A - B).mean()
    knee_corr = np.corrcoef(A[:, 3], B[:, 3])[0, 1]   # column 3 = left knee
    print(f"{a} vs {b}:  mean |difference| {diff:.3f} rad   left-knee correlation {knee_corr:+.2f}")

walk3_subject1 vs walk3_subject2:  mean |difference| 0.324 rad   left-knee correlation +0.02
walk1_subject1 vs walk1_subject2:  mean |difference| 0.271 rad   left-knee correlation +0.05
walk3_subject1 vs walk3_subject4:  mean |difference| 0.310 rad   left-knee correlation +0.30
walk2_subject1 vs walk2_subject3:  mean |difference| 0.310 rad   left-knee correlation +0.45
